[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/reasoning_rl/rft_grpo_math.ipynb)

# Managed GRPO RFT on GSM8K — pure Python SDK

Reinforcement Fine-Tuning where the model learns to solve grade-school math via **GRPO** with a
**verifiable** numeric reward — done entirely through the **Fireworks Python SDK** (`fireworks-ai`),
no `firectl` and no cookbook. Managed RFT means Fireworks runs the rollouts + policy updates for
you; you just supply a dataset and an **evaluator** (the reward).

## Pipeline

1. Build a GSM8K dataset (`{messages, ground_truth}`) from HuggingFace.
2. Define the reward as a **code evaluator** and create it via `client.evaluators.create(...)`
   (the reward code is uploaded inline — no tar, no separate CLI).
3. **Eval BEFORE**: deploy the base model, score GSM8K accuracy, tear it down.
4. `client.reinforcement_fine_tuning_jobs.create(...)` (GRPO) and poll to completion.
5. Deploy the tuned model and **Eval AFTER**; print the before -> after lift.

## Prerequisites

```bash
pip install --pre fireworks-ai "eval-protocol[huggingface]" python-dotenv datasets nest_asyncio
```

`.env`: `FIREWORKS_API_KEY`, `FIREWORKS_ACCOUNT_ID`.

> The SDK is alpha; the RFT/evaluator API may drift. Train + deploy cells provision real GPU and
> **cost money** — keep `GSM8K_ROW_COUNT` / `N_EVAL` small for a first pass.

## 1. Setup & credentials

Imports, repo-root `.env`, the Fireworks SDK client, and shared poll/upload helpers.

In [1]:
import json
import os
import re
import sys
import time
import uuid
from pathlib import Path

import dotenv

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)  # repo-root .env

# Put repo root + common/ on the path so the shared eval helper (ep_eval) imports.
_REPO_ROOT = Path.cwd()
while _REPO_ROOT != _REPO_ROOT.parent and not (_REPO_ROOT / "common" / "ep_eval.py").exists():
    _REPO_ROOT = _REPO_ROOT.parent
for _p in (str(_REPO_ROOT), str(_REPO_ROOT / "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from fireworks import Fireworks, file_from_path

assert os.getenv("FIREWORKS_API_KEY"), "FIREWORKS_API_KEY missing from .env"
ACCOUNT_ID = os.getenv("FIREWORKS_ACCOUNT_ID", "").replace("accounts/", "", 1)
assert ACCOUNT_ID, "FIREWORKS_ACCOUNT_ID missing from .env"

client = Fireworks()  # reads FIREWORKS_API_KEY + FIREWORKS_ACCOUNT_ID from env


def wait_until(get_fn, ok, bad, label, every=30, tries=240):
    obj = None
    for i in range(tries):
        obj = get_fn()
        st = getattr(obj, "state", None)
        print(f"[{label}] {i + 1:03d} state={st}")
        if st in ok:
            return obj
        if st in bad:
            raise RuntimeError(f"{label} failed: state={st}")
        time.sleep(every)
    raise TimeoutError(f"{label} did not finish after {tries} polls")


def upload_dataset(ds_id, path, fmt="CHAT"):
    n_examples = sum(1 for line in open(path, encoding="utf-8") if line.strip())
    client.datasets.create(dataset_id=ds_id, dataset={"format": fmt, "example_count": n_examples})
    client.datasets.upload(dataset_id=ds_id, file=file_from_path(path))
    wait_until(lambda: client.datasets.get(ds_id), {"READY"}, {"STATE_UNSPECIFIED"},
               "dataset", every=3, tries=200)
    return f"accounts/{ACCOUNT_ID}/datasets/{ds_id}"


print("SDK client ready. account:", ACCOUNT_ID)

SDK client ready. account: pyroworks


## 2. Configuration

Base model, GRPO knobs, and sample sizes. `qwen3-8b` is a small, tunable reasoning model; RFT
teaches it to produce answers that pass the verifiable reward.

In [2]:
# CONFIG
BASE_MODEL = "accounts/fireworks/models/qwen3-8b"   # tunable reasoning model
GSM8K_ROW_COUNT = 500        # train rows pulled from HuggingFace
N_EVAL = 20                  # holdout rows scored before/after (GPU deploy each)

# GRPO knobs
EPOCHS = 6
LEARNING_RATE = 1e-4
LORA_RANK = 8
TEMPERATURE = 0.7
COMPLETIONS_PER_PROMPT = 8   # GRPO group size (response candidates per prompt)
MAX_OUTPUT_TOKENS = 512
RL_METHOD = "GRPO"           # also: DAPO, GSPO_TOKEN (see ReinforcementLearningLossConfig)
ACCELERATOR_TYPE = "NVIDIA_H200_141GB"   # SDK deploy needs an explicit accelerator
EVAL_MAX_TOKENS = 1024       # room for step-by-step reasoning before <answer>

TRAIN_FILE = "./gsm8k_train.jsonl"       # regenerated below (gitignored)
DATASET_ID = f"grpo-math-ds-{uuid.uuid4().hex[:6]}"
EVALUATOR_ID = f"grpo-math-eval-{uuid.uuid4().hex[:6]}"
OUTPUT_MODEL_ID = f"qwen3-8b-grpo-math-{uuid.uuid4().hex[:6]}"
DEPLOYMENT_ID = f"grpo-math-{uuid.uuid4().hex[:6]}"

GSM8K_SYSTEM_PROMPT = (
    "You are a helpful assistant that solves math problems step by step. "
    "Show your work and put the final answer in <answer></answer> tags."
)
print("dataset_id:", DATASET_ID, "| evaluator_id:", EVALUATOR_ID, "| output_model_id:", OUTPUT_MODEL_ID)

dataset_id: grpo-math-ds-465dde | evaluator_id: grpo-math-eval-c17baf | output_model_id: qwen3-8b-grpo-math-2ae92e


## 3. Build the GSM8K dataset

Each row is `{"messages": [system, user], "ground_truth": "<final answer>"}` — no assistant turn,
since the model generates that during RL rollouts. We hold out the last `N_EVAL` rows for scoring.

In [3]:
from datasets import load_dataset


def _final_answer(gt: str) -> str:
    # GSM8K answers end with "#### <number>"; keep just the number.
    m = re.search(r"####\s*(.+?)\s*$", gt.strip())
    return m.group(1) if m else gt.strip()


def to_row(ex) -> dict:
    return {
        "messages": [
            {"role": "system", "content": GSM8K_SYSTEM_PROMPT},
            {"role": "user", "content": ex["question"]},
        ],
        "ground_truth": _final_answer(ex["answer"]),
    }


ds = load_dataset("openai/gsm8k", "main", split="train")
rows = [to_row(ds[i]) for i in range(min(GSM8K_ROW_COUNT + N_EVAL, len(ds)))]
train_rows, eval_rows = rows[:-N_EVAL], rows[-N_EVAL:]

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps(r) + "\n")

print(f"wrote {len(train_rows)} train rows -> {TRAIN_FILE}; held out {len(eval_rows)} for eval")
print("\nExample row:")
print(json.dumps(train_rows[0], indent=2)[:500])

  from .autonotebook import tqdm as notebook_tqdm


wrote 500 train rows -> ./gsm8k_train.jsonl; held out 20 for eval

Example row:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant that solves math problems step by step. Show your work and put the final answer in <answer></answer> tags."
    },
    {
      "role": "user",
      "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
    }
  ],
  "ground_truth": "72"
}


## 4. Define & upload the evaluator (the reward)

The reward is a small **code evaluator**: for each rollout it runs `math_reward` (from
`eval-protocol`), scoring 1.0 if the model's `<answer>` matches the ground truth, else 0.0. We
write the code to a folder and upload it with `eval_protocol.create_evaluation` (the V2
upload→build flow), then poll until it finishes building (`BUILDING` → `ACTIVE`).

> Why not `client.evaluators.create`? For a multi-file inline evaluator it sets `BUILDING` but
> doesn't start the build (the build only fires via the upload flow), so it hangs. The
> `create_evaluation` helper does the upload that triggers the build — and it's still pure SDK
> (no `firectl`).

In [4]:
# Write the reward code to a folder and upload it via eval-protocol's V2 flow
# (create_evaluation) — this uploads the code, triggers the build, and reaches ACTIVE.
# NOTE: the fireworks-ai client.evaluators.create with inline multi-file code_snippets
# currently sets state=BUILDING but does NOT start the build (the build only fires via the
# upload->validate flow), so it hangs forever. create_evaluation is the working SDK path.
from eval_protocol.evaluation import create_evaluation

_eval_dir = Path("grpo_math_evaluator")
_eval_dir.mkdir(exist_ok=True)
(_eval_dir / "test_math.py").write_text(
    '''from eval_protocol.models import EvaluateResult, EvaluationRow
from eval_protocol.pytest import evaluation_test, SingleTurnRolloutProcessor
from eval_protocol.rewards.math import math_reward


@evaluation_test(
    input_dataset=["gsm8k_train.jsonl"],
    # completion_params must be present so managed RFT can inject the policy model into it at
    # rollout time (do NOT hardcode "model" — the trainer supplies the model being optimized).
    completion_params=[{"max_tokens": 512, "temperature": 0.7}],
    rollout_processor=SingleTurnRolloutProcessor(),
    mode="pointwise",
)
def evaluate(row: EvaluationRow) -> EvaluationRow:
    """Verifiable math reward: 1.0 if the final <answer> matches ground truth, else 0.0."""
    result = math_reward(messages=row.messages, ground_truth=row.ground_truth)
    row.evaluation_result = EvaluateResult(score=result.score, reason=result.reason)
    return row
'''
)
(_eval_dir / "requirements.txt").write_text("eval-protocol>=0.3.23\nopenai\n")

# create_evaluation tars the current directory, so chdir into the evaluator folder first.
_orig_cwd = os.getcwd()
os.chdir(_eval_dir)
try:
    _res = create_evaluation(
        evaluator_id=EVALUATOR_ID,
        display_name=EVALUATOR_ID,
        description="GSM8K verifiable math reward (SingleTurnRolloutProcessor + math_reward)",
        force=True,
        account_id=ACCOUNT_ID,
        api_key=os.environ["FIREWORKS_API_KEY"],
        entry_point="test_math.py::evaluate",
    )
finally:
    os.chdir(_orig_cwd)

EVALUATOR_NAME = (_res.get("name") if isinstance(_res, dict) else getattr(_res, "name", None)) \
    or f"accounts/{ACCOUNT_ID}/evaluators/{EVALUATOR_ID}"
print("evaluator:", EVALUATOR_NAME)

# Poll until the evaluator finishes building (BUILDING -> ACTIVE); RFT create needs it ACTIVE.
wait_until(lambda: client.evaluators.get(EVALUATOR_ID), {"ACTIVE"},
           {"FAILED", "ERROR", "CANCELLED", "BUILD_FAILED"}, "evaluator", every=10, tries=90)

evaluator: accounts/pyroworks/evaluators/grpo-math-eval-c17baf
[evaluator] 001 state=BUILDING
[evaluator] 002 state=BUILDING
[evaluator] 003 state=BUILDING
[evaluator] 004 state=BUILDING
[evaluator] 005 state=BUILDING
[evaluator] 006 state=ACTIVE


EvaluatorGetResponse(commit_hash='0.0.0.dev3+g9f94342.dirty', created_by='sinan@fireworks.ai', create_time=datetime.datetime(2026, 7, 11, 18, 27, 7, 386612, tzinfo=TzInfo(0)), criteria=[], default_dataset='', description='GSM8K verifiable math reward (SingleTurnRolloutProcessor + math_reward)', display_name='grpo-math-eval-c17baf', entry_point='test_math.py::evaluate', name='accounts/pyroworks/evaluators/grpo-math-eval-c17baf', requirements='', source=None, state='ACTIVE', status=Status(code='OK', message='Template built successfully'), update_time=datetime.datetime(2026, 7, 11, 18, 27, 52, 94696, tzinfo=TzInfo(0)), commitHash='0.0.0.dev3+g9f94342.dirty', createTime='2026-07-11T18:27:07.386612Z', createdBy='sinan@fireworks.ai', defaultDataset='', displayName='grpo-math-eval-c17baf', encryptionState='ENCRYPTION_STATE_PLAINTEXT', entryPoint='test_math.py::evaluate', updateTime='2026-07-11T18:27:52.094696Z')

## 5. Sanity-check the reward (no GPU)

Before spending anything on training, peek at one problem and run the reward locally on a
**correct** vs a **wrong** answer — this is the exact `math_reward` the server-side evaluator
uses during RL rollouts, so if it behaves here it'll behave in training.

In [5]:
# Inspect one held-out problem and sanity-check the reward locally (no GPU).
from eval_protocol.rewards.math import math_reward as _math_reward

_ex = eval_rows[0]
_q = next(m["content"] for m in _ex["messages"] if m["role"] == "user")
_gold = _ex["ground_truth"]
print("Q:", _q[:200])
print("gold answer:", _gold)


def _with_answer(ans):
    # Build a messages list with a candidate assistant answer in <answer></answer>.
    return _ex["messages"] + [{"role": "assistant", "content": f"<answer>{ans}</answer>"}]


_correct = _math_reward(messages=_with_answer(_gold), ground_truth=_gold)
_wrong = _math_reward(messages=_with_answer("-999999"), ground_truth=_gold)
print(f"\nreward(correct={_gold!r}) = {_correct.score}  ({_correct.reason})")
print(f"reward(wrong='-999999')   = {_wrong.score}  ({_wrong.reason})")
assert _correct.score == 1.0 and _wrong.score == 0.0, "reward not behaving as expected!"
print("\nReward sanity check passed: correct -> 1.0, wrong -> 0.0")

Q: Joe played catch with Derek and Tammy. He caught the ball 23 times. Derek made four less than double the catches Joe did. Tammy caught the ball sixteen more than a third of the times Derek did. How ma
gold answer: 30

reward(correct='30') = 1.0  (Best match: Gen='<answer>30</answer>' (30.0) vs Orig='30' (30.0).
Numeric match: Yes, Similarity: 1.000)
reward(wrong='-999999')   = 0.0  (No score match: Gen='<answer>-999999</answer>' (-999999.0) vs Orig='30' (30.0).
Numeric match: No, Similarity: 0.000)

Reward sanity check passed: correct -> 1.0, wrong -> 0.0


  class TaskDefinitionModel(BaseModel):


## 6. Evaluate the base model (before)

Score GSM8K accuracy on the held-out rows through **eval-protocol** (same `math_reward` the
evaluator uses). `qwen3-8b` isn't serverless on all accounts, so we deploy it, score it, and tear
it down.

In [6]:
from ep_eval import single_turn_eval, EvaluationRow, Message, EvaluateResult, final_text
from eval_protocol.rewards.math import math_reward


def build_eval_rows(examples):
    out = []
    for ex in examples:
        msgs = [Message(role=m["role"], content=m["content"]) for m in ex["messages"]]
        out.append(EvaluationRow(messages=msgs, ground_truth=ex["ground_truth"]))
    return out


async def gsm8k_reward(row: EvaluationRow) -> EvaluationRow:
    r = math_reward(messages=row.messages, ground_truth=row.ground_truth)
    row.evaluation_result = EvaluateResult(score=r.score, reason=r.reason)
    return row


def accuracy(model):
    mean, _ = single_turn_eval(build_eval_rows(eval_rows), model, gsm8k_reward,
                               max_tokens=EVAL_MAX_TOKENS, temperature=0.0)
    return mean


def eval_deployed(model_resource, label):
    """Deploy a (non-serverless) model on-demand, score accuracy, then DELETE the deployment."""
    did = f"grpomatheval-{label}-{uuid.uuid4().hex[:6]}"
    dep = client.deployments.create(base_model=model_resource, deployment_id=did,
                                    accelerator_type=ACCELERATOR_TYPE,
                                    min_replica_count=1, max_replica_count=1)
    d_id = dep.name.split("/")[-1]
    try:
        wait_until(lambda: client.deployments.get(d_id), {"READY"}, {"FAILED"},
                   f"deploy-{label}", every=15, tries=160)
        return accuracy(f"accounts/{ACCOUNT_ID}/deployments/{d_id}")
    finally:
        try:
            client.deployments.delete(d_id, ignore_checks=True)
            print(f"[{label}] deployment {d_id} deleted")
        except Exception:  # noqa: BLE001
            try:
                client.deployments.scale(d_id, replica_count=0)
                print(f"[{label}] scaled to 0 (delete blocked): {d_id}")
            except Exception as e2:  # noqa: BLE001
                print(f"[{label}] teardown skip: {str(e2)[:100]}")


print(f"Scoring BASE model on {len(eval_rows)} GSM8K rows (deploy -> eval -> teardown)...")
before = eval_deployed(BASE_MODEL, "base")
print(f"BEFORE accuracy: {before:.1%}")

Scoring BASE model on 20 GSM8K rows (deploy -> eval -> teardown)...
[deploy-base] 001 state=CREATING
[deploy-base] 002 state=CREATING
[deploy-base] 003 state=CREATING
[deploy-base] 004 state=CREATING
[deploy-base] 005 state=CREATING
[deploy-base] 006 state=CREATING
[deploy-base] 007 state=CREATING
[deploy-base] 008 state=CREATING
[deploy-base] 009 state=CREATING
[deploy-base] 010 state=CREATING
[deploy-base] 011 state=CREATING
[deploy-base] 012 state=CREATING
[deploy-base] 013 state=CREATING
[deploy-base] 014 state=CREATING
[deploy-base] 015 state=CREATING
[deploy-base] 016 state=CREATING
[deploy-base] 017 state=CREATING
[deploy-base] 018 state=CREATING
[deploy-base] 019 state=CREATING
[deploy-base] 020 state=CREATING
[deploy-base] 021 state=CREATING
[deploy-base] 022 state=CREATING


11:33:32 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
11:33:32 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
11:33:32 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
11:33:32 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM compl

[deploy-base] 023 state=READY


11:33:37 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
11:33:37 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
11:33:37 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
11:33:38 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpomatheval-base-07f1c2; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM compl

[base] deployment grpomatheval-base-07f1c2 deleted
BEFORE accuracy: 45.0%


## 7. Reinforcement fine-tune with GRPO (GO LIVE)

Upload the dataset and create the managed RFT job. Fireworks runs the rollouts (the model
generates `COMPLETIONS_PER_PROMPT` candidates per prompt), scores them with your evaluator, and
does the GRPO policy update — all server-side. We poll to completion.

**This costs money and GPU quota.**

In [7]:
dataset_name = upload_dataset(DATASET_ID, TRAIN_FILE, fmt="CHAT")
print("dataset:", dataset_name)

_wandb = ({"enabled": True, "api_key": os.getenv("WANDB_API_KEY"), "entity": os.getenv("WANDB_ENTITY"),
           "project": os.getenv("WANDB_PROJECT", "grpo-math-rft-sdk")}
          if os.getenv("WANDB_API_KEY") and os.getenv("WANDB_ENTITY") else None)

job = client.reinforcement_fine_tuning_jobs.create(
    dataset=dataset_name,
    evaluator=EVALUATOR_NAME,
    training_config={
        "base_model": BASE_MODEL,
        "output_model": f"accounts/{ACCOUNT_ID}/models/{OUTPUT_MODEL_ID}",
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lora_rank": LORA_RANK,
    },
    inference_parameters={
        "temperature": TEMPERATURE,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "response_candidates_count": COMPLETIONS_PER_PROMPT,
    },
    loss_config={"method": RL_METHOD},
    eval_auto_carveout=True,
    **({"wandb_config": _wandb} if _wandb else {}),
)
job_id = job.name.split("/")[-1]
print("RFT job:", job.name)
wait_until(lambda: client.reinforcement_fine_tuning_jobs.get(job_id),
           {"JOB_STATE_COMPLETED"}, {"JOB_STATE_FAILED", "JOB_STATE_CANCELLED"}, "rft")
TUNED_MODEL_NAME = f"accounts/{ACCOUNT_ID}/models/{OUTPUT_MODEL_ID}"
print("Tuned model:", TUNED_MODEL_NAME)

[dataset] 001 state=READY
dataset: accounts/pyroworks/datasets/grpo-math-ds-465dde
RFT job: accounts/pyroworks/reinforcementFineTuningJobs/b0zoa67q
[rft] 001 state=JOB_STATE_RUNNING
[rft] 002 state=JOB_STATE_RUNNING
[rft] 003 state=JOB_STATE_RUNNING
[rft] 004 state=JOB_STATE_RUNNING
[rft] 005 state=JOB_STATE_RUNNING
[rft] 006 state=JOB_STATE_RUNNING
[rft] 007 state=JOB_STATE_RUNNING
[rft] 008 state=JOB_STATE_RUNNING
[rft] 009 state=JOB_STATE_RUNNING
[rft] 010 state=JOB_STATE_RUNNING
[rft] 011 state=JOB_STATE_RUNNING
[rft] 012 state=JOB_STATE_RUNNING
[rft] 013 state=JOB_STATE_RUNNING
[rft] 014 state=JOB_STATE_RUNNING
[rft] 015 state=JOB_STATE_RUNNING
[rft] 016 state=JOB_STATE_RUNNING
[rft] 017 state=JOB_STATE_RUNNING
[rft] 018 state=JOB_STATE_RUNNING
[rft] 019 state=JOB_STATE_RUNNING
[rft] 020 state=JOB_STATE_RUNNING
[rft] 021 state=JOB_STATE_RUNNING
[rft] 022 state=JOB_STATE_RUNNING
[rft] 023 state=JOB_STATE_RUNNING
[rft] 024 state=JOB_STATE_RUNNING
[rft] 025 state=JOB_STATE_RUNNING
[r

## 8. Deploy the tuned model

Stand up an on-demand deployment of the RFT'd model so we can score it.

In [8]:
dep = client.deployments.create(
    base_model=TUNED_MODEL_NAME,
    deployment_id=DEPLOYMENT_ID,
    accelerator_type=ACCELERATOR_TYPE,
    min_replica_count=1,
    max_replica_count=1,
)
dep_id = dep.name.split("/")[-1]
wait_until(lambda: client.deployments.get(dep_id), {"READY"}, {"FAILED"}, "deploy", every=15, tries=160)
INFER_MODEL = f"accounts/{ACCOUNT_ID}/deployments/{dep_id}"
print("Deployed. Infer id:", INFER_MODEL)

[deploy] 001 state=CREATING
[deploy] 002 state=CREATING
[deploy] 003 state=CREATING
[deploy] 004 state=CREATING
[deploy] 005 state=CREATING
[deploy] 006 state=CREATING
[deploy] 007 state=CREATING
[deploy] 008 state=CREATING
[deploy] 009 state=CREATING
[deploy] 010 state=CREATING
[deploy] 011 state=CREATING
[deploy] 012 state=CREATING
[deploy] 013 state=READY
Deployed. Infer id: accounts/pyroworks/deployments/grpo-math-315473


## 9. Evaluate the tuned model (after) & compare

Re-run the same GSM8K accuracy on the tuned deployment and print the before -> after lift.

In [9]:
print(f"Scoring TUNED model on {len(eval_rows)} GSM8K rows...")
after = accuracy(INFER_MODEL)

print("\n================ GSM8K accuracy (verifiable reward, via eval-protocol) ================")
print(f"  BEFORE (base) : {before:.1%}")
print(f"  AFTER  (grpo) : {after:.1%}")
print(f"  delta         : {after - before:+.1%}")
print("======================================================================================")
print("PASS: RFT improved accuracy." if after > before else "NO IMPROVEMENT: try more rows/epochs.")

13:14:14 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
13:14:14 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
13:14:14 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
13:14:14 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-

Scoring TUNED model on 20 GSM8K rows...


13:14:36 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
13:14:37 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
13:14:37 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
13:14:37 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-315473; provider = fireworks_ai
INFO:LiteLLM:
LiteLLM completion() model= accounts/pyroworks/deployments/grpo-math-


================ GSM8K accuracy (verifiable reward, via eval-protocol) ================
  BEFORE (base) : 45.0%
  AFTER  (grpo) : 65.0%
  delta         : +20.0%
PASS: RFT improved accuracy.


## 10. Cleanup — tear down the deployment

Delete the tuned model's deployment so it stops billing (safe to re-run).

In [10]:
def _teardown(dep_id):
    try:
        client.deployments.delete(dep_id, ignore_checks=True)
        print("deleted deployment:", dep_id)
    except Exception:  # noqa: BLE001
        try:
            client.deployments.scale(dep_id, replica_count=0)
            print(f"scaled to 0 (delete blocked): {dep_id}")
        except Exception as e2:  # noqa: BLE001
            print(f"skip {dep_id}: {str(e2)[:120]}")


for _dep in [globals().get("DEPLOYMENT_ID")]:
    if _dep:
        _teardown(_dep)

deleted deployment: grpo-math-315473
